In [ ]:
%pylab inline
import eucare as ec

In [ ]:
G = ec.io.load_graph('graphs/hyperbolic_annulus_7_smooth.heg')
G.show()

In [ ]:
GD = G.copy()
GD.add_graph(D)
GD.show(render_faces=False, height=1000)

In [ ]:
dual_pos = {}
for v in D.vertices:
    dual_pos[v['pre']] = v['pos']
    
hs = list(h for h in G.halfedges if not (h.on_border() or h.rev.on_border()))
errors = []
for h in hs:
    v1 = dual_pos[h.face] - dual_pos[h.rev.face]
    v2 = h.orig['pos'] - h.dest['pos']
    errors.append(np.dot(v1, v2) / np.linalg.norm(v1) / np.linalg.norm(v2))
errors = np.array(errors)
errors = np.arccos(errors) - np.pi/2
print(f'Maximum error: {np.max(np.abs(errors))*180/np.pi:.2f}°')

In [ ]:
for h in np.array(hs)[errors > 1e-12]:
    h['color_key'] = (1, 0, 0)
    
G.show()

In [ ]:
from eucare.utils import invert_mapping

def circumcenter(f, eps=1e-6):
    ps = list(v['pos'] for v in f.vertex_iter())
    ax = ps[0][0]
    ay = ps[0][1]
    bx = ps[1][0]
    by = ps[1][1]
    cx = ps[2][0]
    cy = ps[2][1]
    d = 2 * (ax * (by - cy) + bx * (cy - ay) + cx * (ay - by))
    ux = ((ax * ax + ay * ay) * (by - cy) + (bx * bx + by * by) * (cy - ay) + (cx * cx + cy * cy) * (ay - by)) / d
    uy = ((ax * ax + ay * ay) * (cx - bx) + (bx * bx + by * by) * (ax - cx) + (cx * cx + cy * cy) * (bx - ax)) / d
    center = np.array([ux, uy])
    dists = np.linalg.norm(ps - center, axis=-1)
    dists -= np.mean(dists)
    assert np.max(np.abs(dists/np.mean(dists) - 1)) < eps, f'Could not find a circumcenter'
    return center
    

def make_reciprocal_figure(G, dual_positions):
    D, (v_map, e_map, f_map) = G.copy(return_mappings=True)
    dual_positions = {f_map[f]: value for f, value in dual_positions.items()}
    D = ec.conway.dual_graph()(D)
    
    for v in D.vertices:
        v['pos'] = dual_positions[v['pre_conway']]
        
    inv_v_map = invert_mapping(f_map)
    for v in D.vertices:
        v['pre'] = inv_v_map[v['pre_conway']]
    inv_e_map = invert_mapping(e_map)
    for e in D.halfedges:
        if 'pre_conway' in e.attributes:
            e['pre'] = inv_e_map[e['pre_conway']]
    inv_f_map = invert_mapping(v_map)
    for f in D.faces:
        f['pre'] = inv_f_map[f['pre_conway']]
    return D

dual_positions = {f: circumcenter(f) for f in G.faces}
D = make_reciprocal_figure(G, dual_positions)

In [ ]:
D = make_reciprocal_figure(G, dual_positions_new)
GD = G.copy()
GD.add_graph(D)
GD.show(render_faces=False, height=1000)

In [ ]:
from tqdm.auto import tqdm

symmetry_mat = ec.base.rotation_matrix(2 * np.pi / 18)

# for each face, find the symmetric ones

n = 18
alpha = 2 * np.pi / n
rot_mats = ec.base.rotation_matrix(alpha * np.arange(n))
inv_rot_mats = ec.base.rotation_matrix(-alpha * np.arange(n))
for f in G.faces:
    f['midpoint'] = f.midpoint()
    f['radius'] = np.linalg.norm(f['midpoint'])
    f['angle'] = ec.base.angle_to_axis(f['midpoint'])

eps = 1e-6

for f in G.faces:
    if 'equiv_class' in f:
        del f['equiv_class']
        
dual_positions_new = {}
for i, f in enumerate(tqdm(G.faces)):
    if 'equiv_class' in f:
        continue
    equiv_class = [f]
    offsets = [0]
    for f2 in G.faces:
        if f2 is f:
            continue
        if abs(f['radius'] - f2['radius']) > eps:
            continue
        angle_diff = (f['angle'] - f2['angle'] + alpha/2) % alpha - alpha/2
        if np.abs(angle_diff) < eps:
            equiv_class.append(f2)
            offsets.append((int(np.round((f['angle'] - f2['angle'])/alpha)) + n) % n)
    for f2 in equiv_class:
        f2['equiv_class'] = i
    equiv_class = np.array(equiv_class)[np.argsort(offsets)]
    ps = np.stack([dual_pos[f] for f in equiv_class])
    canonical_pos = np.einsum('ni,ikn->nk', ps, rot_mats)
#     plt.scatter(*canonical_pos.T)
#     plt.show()
    canonical_pos = np.median(canonical_pos, axis=0)
    ps_new = np.einsum('i,ikn->nk', canonical_pos, inv_rot_mats)
    for p, f2 in zip(ps_new, equiv_class):
        dual_positions_new[f2] = p

In [ ]:
rot_mats.shape